# 01 - Data Cleaning & Alignment

This notebook loads raw data files and performs:
- Data loading from raw files
- Timestamp alignment across datasets
- Missing value handling
- Unit normalization
- Basic quality checks and exploratory visualizations

## Outputs
- Clean, aligned dataset saved to `data_processed/clean_data.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import sys

# Add src to path
sys.path.append('../')

from src.data import DataLoader, DataCleaner, clean_pipeline
from src.utils import load_config, plot_time_series

# Set display options
pd.set_option('display.max_columns', 50)
sns.set_style('whitegrid')
%matplotlib inline

## 1. Load Configuration

In [ ]:
# Load configuration
config = load_config('../configs/modelling_config.yaml')
print(config)

## 2. Load Raw Data

In [ ]:
# Initialize data loader
data_path = config.get('data.raw_path', '../data_raw')
loader = DataLoader(data_path)

# Load all datasets
try:
    raw_data = loader.load_all_data()
    print(f"\nLoaded {len(raw_data)} datasets")
    for name, df in raw_data.items():
        print(f"  {name}: {len(df)} records, {df.index.min()} to {df.index.max()}")
except Exception as e:
    print(f"Error loading data: {e}")
    print("\nUsing sample data for demonstration...")
    from src.data import load_sample_data
    raw_data = load_sample_data()

## 3. Inspect Raw Data

In [ ]:
# Check each dataset
for name, df in raw_data.items():
    print(f"\n{'='*60}")
    print(f"Dataset: {name}")
    print('='*60)
    print(f"\nShape: {df.shape}")
    print(f"\nColumns: {df.columns.tolist()}")
    print(f"\nMissing values:\n{df.isna().sum()}")
    print(f"\nBasic statistics:")
    print(df.describe())
    
    # Plot
    plt.figure(figsize=(12, 4))
    df.plot()
    plt.title(f'{name.title()} - Raw Data')
    plt.tight_layout()
    plt.show()

## 4. Data Cleaning Pipeline

In [ ]:
# Run cleaning pipeline
freq = config.get('data.freq', 'h')
missing_method = config.get('data.cleaning.missing_method', 'interpolate')
missing_limit = config.get('data.cleaning.missing_limit', 3)

clean_data = clean_pipeline(
    raw_data,
    freq=freq,
    missing_method=missing_method,
    missing_limit=missing_limit
)

print(f"\nClean data shape: {clean_data.shape}")
print(f"\nColumns: {clean_data.columns.tolist()}")
print(f"\nMissing values:\n{clean_data.isna().sum()}")

## 5. Quality Checks

In [ ]:
# Validate clean data
cleaner = DataCleaner()
is_valid, issues = cleaner.validate_data(clean_data)

if is_valid:
    print("✓ Data validation passed!")
else:
    print("⚠ Data validation found issues:")
    for issue in issues:
        print(f"  - {issue}")

## 6. Exploratory Visualization

In [ ]:
# Plot all time series
fig, axes = plt.subplots(len(clean_data.columns), 1, figsize=(14, 3*len(clean_data.columns)))
if len(clean_data.columns) == 1:
    axes = [axes]

for ax, col in zip(axes, clean_data.columns):
    clean_data[col].plot(ax=ax, linewidth=0.5)
    ax.set_title(col.replace('_', ' ').title())
    ax.set_xlabel('')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Summary statistics
print("\nSummary Statistics:")
print(clean_data.describe())

In [ ]:
# Correlation matrix
plt.figure(figsize=(10, 8))
sns.heatmap(clean_data.corr(), annot=True, fmt='.2f', cmap='coolwarm', center=0)
plt.title('Correlation Matrix')
plt.tight_layout()
plt.show()

## 7. Save Clean Data

In [ ]:
# Save to processed data folder
output_path = Path(config.get('data.processed_path', '../data_processed'))
output_path.mkdir(parents=True, exist_ok=True)

output_file = output_path / 'clean_data.csv'
clean_data.to_csv(output_file)

print(f"\n✓ Clean data saved to: {output_file}")
print(f"  Shape: {clean_data.shape}")
print(f"  Date range: {clean_data.index.min()} to {clean_data.index.max()}")

## Summary

Data cleaning complete! The clean dataset is now ready for feature engineering.

**Next steps:**
- Proceed to notebook 02 for feature engineering
- Create residual demand, RES share, and time features
- Add lag and rolling window features